# 10 - Stylometric-neural detector on MAGE

08 built and checked this detector on HC3. This reports it on MAGE (§3.2).

Trained on Viking by `scripts/run_stylometric_detector.py`: eight training sets
by five seeds. Every run scored the whole test split, not only the family it
withheld, so each family can be measured both with and without its own texts in
training.

1. How far does TPR at 1% FPR fall when a family is withheld, and how far does
   AUROC fall on the same texts? (RQ1, H1)
2. Which families drop by more than seed variation, and does the threshold hold
   across the eight training sets?
3. Is any domain much worse than the overall figure? (§3.5)
4. Where does this sit against the lexical baseline and Binoculars?

## 1. Load the scores

Forty runs in one frame, joined to the family and domain of each text.

In [1]:
import numpy as np
import pandas as pd

from logos.paths import DATA_DIR
from logos.evaluation.metrics import (
    calibrate_threshold, tpr_at_threshold, fpr_at_threshold, auroc
)

MAGE_DIR = DATA_DIR / "mage"
SCORES_DIR = MAGE_DIR / "scores" / "stylometric_neural"

meta = pd.read_parquet(
    MAGE_DIR / "corpus.parquet", columns=["text_id", "label", "family", "domain"]
)

scores = pd.concat(
    [pd.read_parquet(path) for path in sorted(SCORES_DIR.glob("*.parquet"))],
    ignore_index=True,
).merge(meta, on="text_id")

runs = scores.groupby(["training_set", "seed"]).ngroups
print(f"{runs} runs, {len(scores):,} scores")
print(scores.groupby("part").size().to_string())

40 runs, 3,732,102 scores
part
calibration    1136280
test           2099000
tune            496822


## 2. All forty runs

Each run is calibrated on its own validation humans rather than on one shared
threshold. The forty runs are forty different models, so a single threshold
would sit at a different false-positive rate for each and the conditions would
not be comparable. Fixing the FPR instead of the threshold is what makes them
comparable, and it is what §3.4 specifies.

In [3]:
def evaluate(run):
    calibration = run[run["part"] == "calibration"]
    test = run[run["part"] == "test"]

    threshold = calibrate_threshold(
        calibration["score"].to_numpy(), calibration["label"].to_numpy()
    )
    human = test[test["label"] == 0]

    rows = []
    for family, machine in test[test["label"] == 1].groupby("family"):
        both = pd.concat([human, machine])
        rows.append({
            "family": family,
            "tpr": tpr_at_threshold(both["score"], both["label"], threshold),
            "auroc": auroc(both["score"], both["label"]),
            "test_fpr": fpr_at_threshold(both["score"], both["label"], threshold),
            "n_machine": len(machine),
        })

    return pd.DataFrame(rows)


results = []
for (set_name, seed), run in scores.groupby(["training_set", "seed"]):
    out = evaluate(run)
    out["training_set"], out["seed"] = set_name, seed
    results.append(out)

results = pd.concat(results, ignore_index=True)

# Three conditions: the family was in training, was the one withheld, or a
# different family was withheld.
results["condition"] = np.select(
    [results["training_set"] == "In-distribution",
     results["training_set"] == "LOFO " + results["family"]],
    ["In-distribution", "LOFO"],
    default="Other LOFO",
)

print(results.groupby("condition").size().to_string())

condition
In-distribution     35
LOFO                35
Other LOFO         210


## 3. Results by family

Each family with its own texts in training, against the run that withheld them.
The six runs that withheld some *other* family are set aside: they vary both the
family withheld and the training data, so they answer no clean question.

A change is interpreted only if it exceeds two standard deviations across the
five seeds (§3.5). The larger of the two conditions' standard deviations is
used, which is the stricter choice.

In [4]:
def summarise(condition):
    g = results[results["condition"] == condition].groupby("family")
    return pd.DataFrame({
        f"tpr_{condition}": g["tpr"].mean(),
        f"sd_{condition}": g["tpr"].std(),
        f"auroc_{condition}": g["auroc"].mean(),
    })


by_family = summarise("In-distribution").join(summarise("LOFO"))
by_family["change"] = by_family["tpr_LOFO"] - by_family["tpr_In-distribution"]
by_family["two_sd"] = 2 * by_family[["sd_In-distribution", "sd_LOFO"]].max(axis=1)
by_family["interpretable"] = by_family["change"].abs() > by_family["two_sd"]

print(f"mean TPR    {by_family['tpr_In-distribution'].mean():.4f} -> {by_family['tpr_LOFO'].mean():.4f}")
print(f"mean AUROC  {by_family['auroc_In-distribution'].mean():.4f} -> {by_family['auroc_LOFO'].mean():.4f}")
print(f"FPR across all runs: {results['test_fpr'].min():.4f} to {results['test_fpr'].max():.4f}")

by_family.sort_values("change").round(4)

mean TPR    0.6649 -> 0.5684
mean AUROC  0.9674 -> 0.9467
FPR across all runs: 0.0086 to 0.0112


,tpr_In-distribution,sd_In-distribution,auroc_In-distribution,tpr_LOFO,sd_LOFO,auroc_LOFO,change,two_sd,interpretable
family,,,,,,,,,
OpenAI GPT,0.6462,0.0275,0.9660,0.3576,0.0188,0.9242,-0.2885,0.0550,True
OPT,0.8540,0.0226,0.9854,0.6442,0.0227,0.9588,-0.2099,0.0454,True
LLaMA,0.7527,0.0287,0.9781,0.6625,0.0478,0.9681,-0.0901,0.0955,False
FLAN-T5,0.1811,0.0517,0.9120,0.1118,0.0250,0.8579,-0.0693,0.1034,False
BigScience,0.4790,0.0447,0.9610,0.4562,0.0225,0.9497,-0.0227,0.0895,False
EleutherAI,0.9380,0.0151,0.9953,0.9402,0.0087,0.9959,0.0022,0.0303,False
GLM-130B,0.8033,0.0260,0.9741,0.8065,0.0138,0.9721,0.0033,0.0519,False


Both metrics fall. H1 is about how differently: the relative fall in TPR against
the relative fall in AUROC.

In [5]:
tpr_fall = 1 - by_family["tpr_LOFO"].mean() / by_family["tpr_In-distribution"].mean()
auroc_fall = 1 - by_family["auroc_LOFO"].mean() / by_family["auroc_In-distribution"].mean()

print(f"TPR falls   {tpr_fall:.1%}")
print(f"AUROC falls {auroc_fall:.1%}")
print(f"ratio       {tpr_fall / auroc_fall:.1f}x")

TPR falls   14.5%
AUROC falls 2.1%
ratio       6.8x


## 4. Results by domain

§3.5 reports by domain so that a large domain cannot hide a failure in a small
one. Each withheld family is measured within each domain against that domain's
human texts, so the false-positive rate is read per domain too.

In [6]:
rows = []
for (set_name, seed), run in scores.groupby(["training_set", "seed"]):
    if set_name == "In-distribution":
        continue

    withheld = set_name.replace("LOFO ", "")
    calibration = run[run["part"] == "calibration"]
    threshold = calibrate_threshold(
        calibration["score"].to_numpy(), calibration["label"].to_numpy()
    )
    test = run[run["part"] == "test"]

    for domain, g in test.groupby("domain"):
        machine = g[(g["label"] == 1) & (g["family"] == withheld)]
        human = g[g["label"] == 0]
        if machine.empty:
            continue
        rows.append({
            "domain": domain, "family": withheld, "seed": seed,
            "tpr": (machine["score"] >= threshold).mean(),
            "fpr": (human["score"] >= threshold).mean(),
            "n_machine": len(machine),
        })

by_domain = pd.DataFrame(rows).groupby("domain").agg(
    tpr=("tpr", "mean"), fpr=("fpr", "mean"), machine_texts=("n_machine", "sum")
)

print(f"overall LOFO TPR {by_family['tpr_LOFO'].mean():.4f}")
by_domain.sort_values("tpr").round(4)

overall LOFO TPR 0.5684


,tpr,fpr,machine_texts
domain,,,
xsum,0.4775,0.0227,13270
eli5,0.5250,0.0085,12985
yelp,0.5300,0.0079,12735
squad,0.5492,0.0087,12475
hswag,0.5806,0.0008,12670
roct,0.5916,0.0006,12940
tldr,0.6036,0.0105,9905
sci_gen,0.6072,0.0129,11255
cmv,0.6126,0.0121,10190


## 5. Against the other detectors

All three under shift, on the same test texts at the same 1% FPR.

Binoculars is untrained, so it has one value per family and no seed variation.
Any comparison with it carries uncertainty on one side only, and §3.5 applies
the two-standard-deviation test using the trained detector's spread alone.

In [7]:
lexical = pd.read_csv(DATA_DIR / "results" / "summary_lexical.csv")
binoculars = pd.read_csv(DATA_DIR / "results" / "summary_binoculars.csv")

comparison = pd.DataFrame({
    "lexical": lexical[lexical["condition"] == "LOFO"].set_index("family")["tpr_mean"],
    "binoculars": binoculars.set_index("family")["tpr_mean"],
    "stylometric_neural": by_family["tpr_LOFO"],
})
comparison["best"] = comparison.idxmax(axis=1)

print(comparison[["lexical", "binoculars", "stylometric_neural"]].mean().round(4).to_string())
comparison.round(4)

lexical        0.1972
binoculars     0.1093
stylometric    0.5684


,lexical,binoculars,stylometric,best
family,,,,
BigScience,0.2360,0.0178,0.4562,stylometric
EleutherAI,0.3985,0.0007,0.9402,stylometric
FLAN-T5,0.0704,0.0163,0.1118,stylometric
GLM-130B,0.2555,0.3090,0.8065,stylometric
LLaMA,0.1025,0.1291,0.6625,stylometric
OPT,0.2157,0.0004,0.6442,stylometric
OpenAI GPT,0.1016,0.2918,0.3576,stylometric


## 6. Save

Summary for Chapter 4, in the same shape as the lexical and Binoculars files.

In [8]:
summary = results.groupby(["family", "condition"]).agg(
    tpr_mean=("tpr", "mean"), tpr_std=("tpr", "std"),
    auroc_mean=("auroc", "mean"), auroc_std=("auroc", "std"),
    test_fpr_mean=("test_fpr", "mean"), test_fpr_std=("test_fpr", "std"),
).reset_index()
summary.insert(0, "detector", "stylometric_neural")

out_path = DATA_DIR / "results" / "summary_stylometric.csv"
summary.round(4).to_csv(out_path, index=False)

print(f"saved {len(summary)} rows to {out_path}")
summary.round(4)

saved 21 rows to /Users/xwm517/Desktop/logos/data/results/summary_stylometric.csv


,detector,family,condition,tpr_mean,tpr_std,auroc_mean,auroc_std,test_fpr_mean,test_fpr_std
0,stylometric,BigScience,In-distribution,0.4790,0.0447,0.9610,0.0019,0.0096,0.0005
1,stylometric,BigScience,LOFO,0.4562,0.0225,0.9497,0.0015,0.0099,0.0003
2,stylometric,BigScience,Other LOFO,0.5418,0.0804,0.9654,0.0068,0.0097,0.0005
3,stylometric,EleutherAI,In-distribution,0.9380,0.0151,0.9953,0.0008,0.0096,0.0005
4,stylometric,EleutherAI,LOFO,0.9402,0.0087,0.9959,0.0007,0.0101,0.0008
5,stylometric,EleutherAI,Other LOFO,0.9375,0.0404,0.9951,0.0039,0.0097,0.0005
6,stylometric,FLAN-T5,In-distribution,0.1811,0.0517,0.9120,0.0053,0.0096,0.0005
7,stylometric,FLAN-T5,LOFO,0.1118,0.0250,0.8579,0.0035,0.0093,0.0002
8,stylometric,FLAN-T5,Other LOFO,0.2572,0.0798,0.9243,0.0114,0.0098,0.0005
9,stylometric,GLM-130B,In-distribution,0.8033,0.0260,0.9741,0.0021,0.0096,0.0005
